In [141]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}
enum class BudgetFactor(val string: String, val value: Int) {
    BUDGET_30("0.3", 30),
    BUDGET_50("0.5", 50),
    BUDGET_70("0.7", 70),
    BUDGET_100("1.0", 100)
}
enum class Parameter(val value: String) {
    CLUSTER_ELIMINATION_THRESHOLD("clustereliminationthreshold"),
    ALPHA("clustereliminationrevenueweight"),
    BETA("clustereliminationsparsityweight"),
}

val percentageFraction = 1
val colsWithoutPercentages = "0.50"
val gradient = 0.3

val parameter = Parameter.CLUSTER_ELIMINATION_THRESHOLD
val mode = Mode.FLAT
val algorithm = Algorithm.OP
val budgetFactor = BudgetFactor.BUDGET_100

val fileName = when (parameter) {
    Parameter.ALPHA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_ea4opml_${parameter.value}_0.0_1.25_0.25.csv"
    Parameter.CLUSTER_ELIMINATION_THRESHOLD -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_ea4opml_${parameter.value}_0.3_0.7_0.1.csv"
    Parameter.BETA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_ea4opml_${parameter.value}_-0.25_1.25_0.25.csv"
}

val relativePath = "/op-solver-strict/results/elimination/${algorithm.name.lowercase()}/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df = df.remove { df.columns()[1] }
df

name,0.30,0.40,0.50,0.60,0.70
eil101,58.000000,59.000000,59.000000,59.000000,60.000000
gil262,134.000000,140.000000,137.000000,137.000000,140.000000
pr299,143.000000,148.000000,150.000000,137.000000,148.000000
lin318,166.000000,183.000000,183.000000,183.000000,180.000000
rd400,202.000000,208.000000,192.000000,205.000000,198.000000
d493,314.000000,302.000000,302.000000,307.000000,292.000000
u574,286.000000,307.000000,309.000000,287.000000,283.000000
u724,356.000000,373.000000,384.000000,366.000000,371.000000
pcb1173,566.000000,579.000000,574.000000,531.000000,495.000000
fl1400,639.000000,835.000000,984.000000,991.000000,875.000000


In [142]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Double).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[60, 140, 150, 183, 208, 314, 309, 384, 579, 991, 1221]

In [143]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, (df["0.50"][index] as Double).toInt()) }

rowMaxValues

[60, 140, 150, 183, 208, 314, 309, 384, 579, 991, 1221]

In [144]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Double>()
}.map { row -> row!!.toInt()}
rowMinValues

[58, 134, 137, 166, 192, 292, 283, 356, 495, 639, 992]

In [145]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Double>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
        calculatePercentage(refValue, (col[row] as Double).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -5.7\%, -0.7\%, -, -2.3\%, -4.9\%]

In [146]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

name,0.30,0.40,0.50,0.60,0.70
eil101,\cellcolor{cyan!88} 58{\tiny-1.7\%},\cellcolor{cyan!94} 59{\tiny+0.0\%},\cellcolor{cyan!94} 59,\cellcolor{cyan!94} 59{\tiny+0.0\%},\cellcolor{cyan!100} \textbf{60*}{\ti...
gil262,\cellcolor{cyan!85} 134{\tiny-2.2\%},\cellcolor{cyan!100} \textbf{140*}{\t...,\cellcolor{cyan!92} 137,\cellcolor{cyan!92} 137{\tiny+0.0\%},\cellcolor{cyan!100} \textbf{140*}{\t...
pr299,\cellcolor{cyan!84} 143{\tiny-4.7\%},\cellcolor{cyan!95} 148{\tiny-1.3\%},\cellcolor{cyan!100} \textbf{150*},\cellcolor{cyan!71} 137{\tiny-8.7\%},\cellcolor{cyan!95} 148{\tiny-1.3\%}
lin318,\cellcolor{cyan!69} 166{\tiny-9.3\%},\cellcolor{cyan!100} \textbf{183*}{\t...,\cellcolor{cyan!100} \textbf{183*},\cellcolor{cyan!100} \textbf{183*}{\t...,\cellcolor{cyan!94} 180{\tiny-1.6\%}
rd400,\cellcolor{cyan!90} 202{\tiny+5.2\%},\cellcolor{cyan!100} \textbf{208*}{\t...,\cellcolor{cyan!74} 192,\cellcolor{cyan!95} 205{\tiny+6.8\%},\cellcolor{cyan!83} 198{\tiny+3.1\%}
d493,\cellcolor{cyan!100} \textbf{314*}{\t...,\cellcolor{cyan!87} 302{\tiny+0.0\%},\cellcolor{cyan!87} 302,\cellcolor{cyan!92} 307{\tiny+1.7\%},\cellcolor{cyan!76} 292{\tiny-3.3\%}
u574,\cellcolor{cyan!75} 286{\tiny-7.4\%},\cellcolor{cyan!97} 307{\tiny-0.6\%},\cellcolor{cyan!100} \textbf{309*},\cellcolor{cyan!76} 287{\tiny-7.1\%},\cellcolor{cyan!71} 283{\tiny-8.4\%}
u724,\cellcolor{cyan!75} 356{\tiny-7.3\%},\cellcolor{cyan!90} 373{\tiny-2.9\%},\cellcolor{cyan!100} \textbf{384*},\cellcolor{cyan!84} 366{\tiny-4.7\%},\cellcolor{cyan!88} 371{\tiny-3.4\%}
pcb1173,\cellcolor{cyan!92} 566{\tiny-1.4\%},\cellcolor{cyan!100} \textbf{579*}{\t...,\cellcolor{cyan!97} 574,\cellcolor{cyan!72} 531{\tiny-7.5\%},\cellcolor{cyan!51} 495{\tiny-13.8\%}
fl1400,\cellcolor{cyan!0} 639{\tiny-35.1\%},\cellcolor{cyan!47} 835{\tiny-15.1\%},\cellcolor{cyan!97} 984,\cellcolor{cyan!100} \textbf{991*}{\t...,\cellcolor{cyan!60} 875{\tiny-11.1\%}


In [147]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{1.9cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:$shortAlgString:${mode.name.lowercase()}:${budgetFactor.value}"
val title = "Parameter search: \$R'\$ with \\textit{$shortAlgString}, ${mode.name.lowercase()}, \$ \\gamma = ${budgetFactor.string}\$."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Parameter run for \$R'\$ using $mediumAlgString with $\\gamma = ${budgetFactor.string}\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|  }
                \hline
                \multicolumn{6}{|c|}{Parameter search: $R'$ with \textit{OP}, flat, $ \gamma = 1.0$.} \\
                \hline
                    name & 0.30 & 0.40 & 0.50 & 0.60 & 0.70 \\
                \hline
                    eil101 & \cellcolor{cyan!88} 58{\tiny-1.7\%} & \cellcolor{cyan!94} 59{\tiny+0.0\%} & \cellcolor{cyan!94} 59 & \cellcolor{cyan!94} 59{\tiny+0.0\%} & \cellcolor{cyan!100} \textbf{60*}{\tiny+1.7\%} \\ 
gil262 & \cellcolor{cyan!85} 134{\tiny-2.2\%} & \cellcolor{cyan!100} \textbf{140*}{\tiny+2.2\%} & \cellcolor{cyan!92} 137 & \cellcolor{cyan!92} 137{\tiny+0.0\%} & \cellcolor{cyan!100} \textbf{140*}{\tiny+2.2\%} \\ 
pr299 & \cellcolor{cyan!84} 143{\tiny-4.7\%} & \cellcolor{cyan!95} 148{\tiny-1.3\%} & \cellcolor{cyan!100} \textbf{150*} & \cellcolor{cyan!71} 137{\tiny-8.7\%} & \ce